In [1]:
import numpy as np
import pandas as pd
import sklearn

In [2]:
df_original = pd.read_csv(r'E:\Learnova\Project 2 Housing Price Predition using LR\notebooks\data\Housing.csv')

In [3]:
#Dropping target column
X= df_original.drop('price',axis=1)
y=df_original['price']

In [4]:
import pandas as pd


def detect_outlier(data: pd.DataFrame, column: str) -> pd.DataFrame:
    """Detects and returns rows containing outliers for a specific column using the IQR method."""
    # 1. Calculate quantiles specifically for the targeted column
    q1 = data[column].quantile(0.25)
    q3 = data[column].quantile(0.75)
    IQR = q3 - q1

    # 2. Establish the outlier boundaries
    lower_fence = q1 - (IQR * 1.5)
    upper_fence = q3 + (IQR * 1.5)

    # 3. Filter for values that fall OUTSIDE either fence
    outlier_mask = (data[column] < lower_fence) | (data[column] > upper_fence)
    outlier = data[outlier_mask]

    # 4. Return just the column dataframe as requested
    return outlier[[column]]

In [5]:
# 1. Get a list of only the numeric columns to prevent string crashes
numeric_cols = X.select_dtypes(include=["number"]).columns

# 2. Loop through only the numeric columns
for col in numeric_cols:
    outliers = detect_outlier(X, col)

    # Print a clean summary for each column
    print(f"=== Outliers detected in '{col}' ===")
    if outliers.empty:
        print("No outliers found. Clean!\n")
    else:
        print(f"Total Outliers: {len(outliers)}")
        print(outliers.head(), "\n")  # Shows the first few outlier values

=== Outliers detected in 'area' ===
Total Outliers: 12
     area
7   16200
10  13200
56  11440
64  11175
66  13200 

=== Outliers detected in 'bedrooms' ===
Total Outliers: 12
     bedrooms
7           5
28          5
34          5
89          5
112         6 

=== Outliers detected in 'bathrooms' ===
Total Outliers: 1
   bathrooms
1          4 

=== Outliers detected in 'stories' ===
Total Outliers: 41
    stories
1         4
6         4
9         4
17        4
26        4 

=== Outliers detected in 'parking' ===
Total Outliers: 12
     parking
1          3
3          3
47         3
93         3
225        3 



In [6]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from feature_engine.outliers import Winsorizer

In [7]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=40)

In [8]:
X_train

,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
6,8580,4,3,4,yes,no,no,no,yes,2,yes,semi-furnished
381,4000,2,1,1,yes,no,no,no,no,0,yes,semi-furnished
2,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
166,7800,3,1,1,yes,no,yes,no,yes,2,yes,unfurnished
503,4000,3,1,1,yes,no,no,no,no,0,no,semi-furnished
...,...,...,...,...,...,...,...,...,...,...,...,...
440,3640,4,1,2,yes,no,yes,no,no,0,no,unfurnished
165,6450,3,2,1,yes,yes,yes,yes,no,0,no,unfurnished
7,16200,5,3,2,yes,no,no,no,no,0,no,unfurnished
219,7000,3,1,2,yes,no,yes,no,no,0,no,unfurnished


In [9]:
X_test

,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
42,6480,3,2,4,yes,no,no,no,yes,2,no,unfurnished
62,6240,4,2,2,yes,no,no,no,yes,1,no,furnished
0,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
422,3720,2,1,1,no,no,no,no,yes,0,no,unfurnished
112,4300,6,2,2,yes,no,no,no,no,0,no,furnished
...,...,...,...,...,...,...,...,...,...,...,...,...
360,4040,2,1,1,yes,no,no,no,no,0,no,semi-furnished
85,8250,3,2,3,yes,no,no,no,yes,0,no,furnished
173,5300,4,2,1,yes,no,no,no,yes,0,yes,unfurnished
295,2325,3,1,2,no,no,no,no,no,0,no,semi-furnished


In [10]:
y_train

6      10150000
381     3605000
2      12250000
166     5320000
503     2660000
         ...   
440     3234000
165     5383000
7      10150000
219     4795000
326     3990000
Name: price, Length: 381, dtype: int64

In [11]:
y_test

42      7700000
62      7070000
0      13300000
422     3360000
112     6083000
         ...   
360     3710000
85      6510000
173     5250000
295     4200000
151     5565000
Name: price, Length: 164, dtype: int64

In [12]:
ohe= OneHotEncoder()
scaler=StandardScaler()
winsor = Winsorizer()

In [13]:
numeric_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = X.select_dtypes(include=["str", "category"]).columns.tolist()

In [14]:
numerical_pipeline=Pipeline(
    steps=[('Outlier_Handler',Winsorizer(capping_method='iqr',fold=1.5)),
           ('num_sacaler',StandardScaler())
           ])

In [15]:
catagorical_pipeline =Pipeline(
    steps=[('ohe',OneHotEncoder(drop='first',handle_unknown='error'))

    ]
)

In [16]:
preprocessor = ColumnTransformer(
    transformers=[('numeric_branch',numerical_pipeline,numeric_cols),
                  ('catagorical_branch',catagorical_pipeline,categorical_cols)

    ]
)

In [18]:
X_train = pd.DataFrame(preprocessor.fit_transform(X_train),
                       columns=preprocessor.get_feature_names_out())

In [19]:
X_test = pd.DataFrame(preprocessor.transform(X_test),
                      columns=preprocessor.get_feature_names_out())

In [20]:
from sklearn.linear_model import LinearRegression,SGDRegressor

In [23]:
sgd_reg= SGDRegressor(max_iter=1000,eta0=0.02,random_state=40)

In [24]:
sgd_model = sgd_reg.fit(X_train,y_train)

In [25]:
y_test_pred = sgd_reg.predict(X_test)
y_train_pred = sgd_reg.predict(X_train)

In [26]:
from sklearn.metrics import r2_score,mean_squared_error


In [27]:
r2 = r2_score
r2(y_test,y_test_pred)

0.6346716635492857

In [28]:
r2(y_train,y_train_pred)

0.693552913851906

In [29]:
print(f"Training R² Score: {r2(y_train, y_train_pred):.4f}")
print(f"Testing R² Score:  {r2(y_test, y_test_pred):.4f}")

Training R² Score: 0.6936
Testing R² Score:  0.6347


In [31]:
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import r2_score

# Test different regularization strengths for SGD
alpha_options = [0.0001, 0.001, 0.01, 0.05, 0.1, 0.5]

for a in alpha_options:
    # Initialize SGD with current alpha value
    sgd = SGDRegressor(penalty="l2", alpha=a, max_iter=2000, random_state=42)

    # Train the model
    sgd.fit(X_train, y_train)

    # Predict
    train_preds = sgd.predict(X_train)
    test_preds = sgd.predict(X_test)

    # Evaluate
    print(f"--- SGD with Alpha = {a} ---")
    print(f"Training R²: {r2_score(y_train, train_preds):.4f}")
    print(f"Testing R²:  {r2_score(y_test, test_preds):.4f}\n")

--- SGD with Alpha = 0.0001 ---
Training R²: 0.6940
Testing R²:  0.6336

--- SGD with Alpha = 0.001 ---
Training R²: 0.6940
Testing R²:  0.6336

--- SGD with Alpha = 0.01 ---
Training R²: 0.6940
Testing R²:  0.6331

--- SGD with Alpha = 0.05 ---
Training R²: 0.6894
Testing R²:  0.6273

--- SGD with Alpha = 0.1 ---
Training R²: 0.6821
Testing R²:  0.6162

--- SGD with Alpha = 0.5 ---
Training R²: 0.6250
Testing R²:  0.5469



In [ ]:
# 1. Transform the targets to logarithmic scale
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

# 2. Re-run your SGD model on the log targets
sgd_log = SGDRegressor(
    learning_rate="adaptive", eta0=0.01, max_iter=5000, random_state=40
)
sgd_log.fit(X_train, y_train_log)

# 3. Predict on log scale
train_preds_log = sgd_log.predict(X_train)
test_preds_log = sgd_log.predict(X_test)

# 4. Evaluate on the log scale directly to see if the gap closes
print("=== Log-Transformed Target Performance ===")
print(f"Training R² (Log): {r2_score(y_train_log, train_preds_log):.4f}")
print(f"Testing R² (Log):  {r2_score(y_test_log, test_preds_log):.4f}")

=== Log-Transformed Target Performance ===
Training R² (Log): 0.6935
Testing R² (Log):  0.6846
